# 从已有 LakeSoul 表构建向量索引（S3 / MinIO）

本教程面向这个场景：

- LakeSoul 表**已经存在**，数据已经写在 S3 / MinIO；
- 表在创建时已经声明了 `vector_index_columns`，但历史数据还没有索引
  （例如批量导入时用了 `auto_build_vector_index=False`，或数据是外部流程写入的）；
- 你知道表的 S3 路径，想确认索引文件写到哪里，然后查询。

本 notebook 会：

1. 准备一张“有数据、但还没建索引”的演示表（已有表可跳过）；
2. 打开已有表，确认主键、向量列、`vector_index_columns` 和 **S3 路径**；
3. 用 `build_vector_index()` 给历史数据补建索引（已有索引时改用 `rebuild_vector_index()`）；
4. 查看 `{table.path}/_vector_index/...` 下的索引文件；
5. 用 Python `table.search` 和 DataFusion SQL 查询。

> 前提：表必须有主键；向量列必须是 `float32` 定长列表；表属性里必须有
> `vector_index_columns`。当前 Python API 没有“给已有表追加索引声明”的 alter 入口，
> 如果建表时没声明，需要重建表或在 SQL DDL 中声明。

In [1]:
import os
from pathlib import Path

# ============ MinIO / S3 配置（按你的环境修改） ============
BUCKET = os.environ.get("MINIO_BUCKET", "lakesoul-tutorial")
ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://127.0.0.1:9000")
ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
OPTIONS = {
    "fs.s3a.bucket": BUCKET,
    "fs.s3a.endpoint": ENDPOINT,
    "fs.s3a.access.key": ACCESS_KEY,
    "fs.s3a.secret.key": SECRET_KEY,
    "fs.s3a.path.style.access": "true",  # MinIO 需要 path-style
}

# ============ 向量缓存（可选；必须在 import lakesoul 之前设置） ============
CACHE_ROOT = Path(os.environ.get("LAKESOUL_TUTORIAL_CACHE", "/tmp/lakesoul-existing-table"))
os.environ.setdefault("LAKESOUL_VECTOR_CACHE", "1")
os.environ.setdefault("LAKESOUL_VECTOR_MEMORY_BYTES", "512MiB")
os.environ.setdefault("LAKESOUL_CACHE_PATH", str(CACHE_ROOT / "cache"))
os.environ.setdefault("LAKESOUL_CACHE_SIZE", "1GiB")
os.environ.setdefault("LAKESOUL_INDEX_SSD_BYTES", "256MiB")
os.environ.setdefault("LAKESOUL_VECTOR_LAZY_INDEX", "auto")

from lakesoul import LakeSoulCatalog

# 已有表名 / namespace / S3 前缀
DEMO_TABLE = "tutorial_existing_docs"
TABLE = os.environ.get("LAKESOUL_TABLE", DEMO_TABLE)
NAMESPACE = os.environ.get("LAKESOUL_NAMESPACE", "default")
PREFIX = os.environ.get("LAKESOUL_TABLE_PREFIX", "tutorial/existing-table")
TABLE_PATH = f"s3://{BUCKET}/{PREFIX}"

# 只有默认演示表才允许第 1 节 drop/create；指向已有表时自动跳过，避免误删。
IS_DEMO = TABLE == DEMO_TABLE and NAMESPACE == "default"

catalog = LakeSoulCatalog.from_env(
    namespace=NAMESPACE,
    object_store_options=OPTIONS,
)

print("table:", f"{NAMESPACE}.{TABLE}")
print("demo s3 path:", TABLE_PATH)

table: default.tutorial_existing_docs
demo s3 path: s3://lakesoul-tutorial/tutorial/existing-table


## 1. 准备演示表（如果你已经有表，可跳过本节）

为了让教程可以独立运行，这里创建一张演示表：

- 建表时声明 `vector_index_columns`；
- 写入数据时设置 `auto_build_vector_index=False`，**故意不建索引**；
- 后面再把它当成“已有的、有数据但没索引的表”来补建索引。

真实场景中，你可能已经通过 SQL / Python / 其他引擎写好了数据，只需要从第 2 节开始。

> 使用已有表时：在启动 kernel 前设置 `LAKESOUL_TABLE=<表名>`（可选
> `LAKESOUL_NAMESPACE`），然后跳过本节。第 2 节会从元数据读取真实的 `table.path`。


In [2]:
if not IS_DEMO:
    print("当前使用已有表，跳过演示表创建；直接运行第 2 节。")
else:
    import numpy as np
    import pyarrow as pa

    DIM = 8
    ROWS = [
        [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.9, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        [0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    ]

    schema = pa.schema([
        pa.field("id", pa.int64(), nullable=False),
        pa.field("title", pa.string(), nullable=False),
        pa.field("lang", pa.string(), nullable=False),
        pa.field("embedding", pa.list_(pa.field("item", pa.float32()), DIM), nullable=False),
    ])

    rows = pa.table({
        "id": pa.array(range(1, len(ROWS) + 1), type=pa.int64()),
        "title": pa.array([f"doc-{i}" for i in range(1, len(ROWS) + 1)]),
        "lang": pa.array(["zh", "zh", "en", "en", "zh", "en"]),
        "embedding": pa.FixedSizeListArray.from_arrays(
            pa.array(np.asarray(ROWS, dtype=np.float32).reshape(-1), type=pa.float32()),
            DIM,
        ),
    })

    catalog.drop_table(TABLE, if_exists=True)
    table = catalog.create_table(
        TABLE,
        path=TABLE_PATH,
        schema=schema,
        primary_keys=["id"],
        hash_bucket_num=2,
        vector_index=[{
            "column": "embedding",
            "dim": DIM,
            "nlist": 2,
            "total_bits": 1,
            "metric": "L2",
        }],
    )

    # 只写数据，不建索引：模拟“已有数据、还没索引”的表
    result = table.write_arrow(rows, auto_build_vector_index=False)
    print("write files:", [f.path for f in result.files])
    print("rows written:", result.row_count)


write files: ['s3://lakesoul-tutorial/tutorial/existing-table/part-BfOg8ABw0EjnP0ht_0000.vortex', 's3://lakesoul-tutorial/tutorial/existing-table/part-BfOg8ABw0EjnP0ht_0001.vortex']
rows written: 6


## 2. 打开已有表，确认 S3 路径和索引声明

无论表是刚创建的还是外部已经存在的，都先用逻辑表名把它打开：

```python
table = catalog.table(TABLE)
```

然后确认四件事：

1. `table.path`：表的 S3 根路径；
2. `table.primary_keys`：向量索引要求有主键；
3. `table.vector_index_columns`：必须已经声明向量列；
4. `table.hash_bucket_num`：每个 bucket 一个独立的索引 shard。

索引文件的位置是：

```text
{table.path}/_vector_index/{vector_column}/{partition_desc}/{bucket_id}/
```

未分区表的 `partition_desc` 是 `-5`。

> 如果你只知道 S3 路径、不知道表名，可以先遍历元数据里的表并比对路径：
>
> ```python
> target_path = "s3://your-bucket/your-prefix"
> matches = [
>     name for name in catalog.list_tables(NAMESPACE)
>     if catalog.table(name).path.rstrip("/") == target_path.rstrip("/")
> ]
> ```
>
> 注意：LakeSoul 的索引写入和查询都依赖元数据（表名 + namespace），
> 不能只凭一个 S3 路径直接建索引；S3 路径用于确认数据/索引文件的实际位置。


In [3]:
from pyarrow.fs import FileSelector, FileType, S3FileSystem

table = catalog.table(TABLE)

print("table       :", f"{table.namespace}.{table.name}")
print("s3 path     :", table.path)
print("primary keys:", table.primary_keys)
print("hash buckets:", table.hash_bucket_num)
print("vector cols :", table.vector_index_columns)
print("properties  :", dict(table.properties))

if not table.vector_index_columns:
    raise RuntimeError(
        "表没有声明 vector_index_columns；当前 Python API 没有 alter property 入口，"
        "需要在建表时声明（或通过 SQL DDL 声明后重建表）。"
    )

INDEX_ROOT = f"{table.path.rstrip('/')}/_vector_index"
print("index root  :", INDEX_ROOT)


def split_s3(path):
    """s3://bucket/prefix -> (bucket, prefix)"""
    rest = path.split("://", 1)[1]
    bucket, _, prefix = rest.partition("/")
    return bucket, prefix.strip("/")


TABLE_BUCKET, TABLE_PREFIX = split_s3(table.path)


def s3_fs():
    return S3FileSystem(
        access_key=ACCESS_KEY,
        secret_key=SECRET_KEY,
        endpoint_override=ENDPOINT,
        scheme="http" if ENDPOINT.startswith("http://") else "https",
        region="us-east-1",
    )


def list_s3(location=None):
    location = location or f"{TABLE_BUCKET}/{TABLE_PREFIX}"
    selector = FileSelector(location, recursive=True, allow_not_found=True)
    return [
        info.path
        for info in s3_fs().get_file_info(selector)
        if info.type != FileType.Directory
    ]


def list_index_s3():
    return list_s3(f"{TABLE_BUCKET}/{TABLE_PREFIX}/_vector_index")


index_before = list_index_s3()
print("\nindex objects before:", len(index_before))


table       : default.tutorial_existing_docs
s3 path     : s3://lakesoul-tutorial/tutorial/existing-table
primary keys: ('id',)
hash buckets: 2
vector cols : ('embedding',)
properties  : {'vector_index_columns': '[{"column": "embedding", "dim": 8, "nlist": 2, "total_bits": 1, "metric": "L2"}]', 'hashBucketNum': '2'}
index root  : s3://lakesoul-tutorial/tutorial/existing-table/_vector_index

index objects before: 0


## 3. 给已有数据建索引

两个入口的区别：

| 方法 | 用途 | 是否重训 | 是否幂等 |
|---|---|---|---|
| `table.build_vector_index()` | 历史数据**首次补建**；或对未覆盖文件做增量构建 | 否（沿用已有 centroids） | 已有索引时再调用可能重复追加，不建议 |
| `table.rebuild_vector_index()` | 从全部 active 数据文件**从零重建**；改 `nlist`/`total_bits`/`metric` 后用 | 是 | 是，每次发布新 generation |

下面的代码会先检查 S3 上是否已有 `_vector_index`：

- 没有 → `build_vector_index()`（本教程的首次补建）；
- 有 → `rebuild_vector_index()`（避免重复追加，方便重复运行 notebook）。

In [4]:
import time

existing = list_index_s3()

started = time.perf_counter()
if existing:
    print(f"检测到已有索引文件 {len(existing)} 个，改用 rebuild_vector_index() 幂等重建")
    build_result = table.rebuild_vector_index()
else:
    print("未检测到索引，调用 build_vector_index() 补建历史数据")
    build_result = table.build_vector_index()
print(f"index build/rebuild: {time.perf_counter() - started:.2f}s")
print(build_result)

未检测到索引，调用 build_vector_index() 补建历史数据
  Reservoir: 3 / 3 vectors (0 MB)
  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
  Reservoir: 3 / 3 vectors (0 MB)
  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
index build/rebuild: 0.11s
{'status': 'ok', 'table_name': 'tutorial_existing_docs', 'partitions_total': 1, 'partitions_processed': 1, 'details': [{'status': 'ok', 'shards_tot

## 4. 查看 S3 上的索引文件

索引和表数据在同一个 S3 根路径下：

```text
s3://{bucket}/{table_prefix}/
├── part-*.vortex                     # 数据文件
└── _vector_index/
    └── embedding/
        └── -5/                       # 未分区表；分区表是 range=...
            ├── 0/                    # hash bucket 0
            │   ├── LATEST
            │   ├── cluster_0000_0000_*.seg
            │   └── ...
            └── 1/
                └── ...
```

`build_vector_index` 只会写索引文件，不会把索引数据写回表数据文件。

In [5]:
index_after = list_index_s3()

print("index root:", INDEX_ROOT)
print("index objects:", len(index_after))
for path in sorted(index_after)[:12]:
    print("  ", path)
if len(index_after) > 12:
    print("  ...")

index root: s3://lakesoul-tutorial/tutorial/existing-table/_vector_index
index objects: 4
   lakesoul-tutorial/tutorial/existing-table/_vector_index/embedding/-5/0/cluster_0000_0000_b99f025714dd0052.seg
   lakesoul-tutorial/tutorial/existing-table/_vector_index/embedding/-5/0/cluster_0001_0000_aa0449c0f9d11458.seg
   lakesoul-tutorial/tutorial/existing-table/_vector_index/embedding/-5/1/cluster_0000_0000_38ba9b4a6c3c2f9a.seg
   lakesoul-tutorial/tutorial/existing-table/_vector_index/embedding/-5/1/cluster_0001_0000_1d734c77dbf2cc66.seg


## 5. Python 查询

索引建好后，`table.search` 会走 `LakeSoulVectorSearchExec`：

- `rerank="exact"`：索引召回候选，再读候选行原始向量做精确重排；
- `rerank="index"`：只按索引距离排序，不读向量列。

查询 session 需要带上对象存储配置和 `warehouse_prefix`。

In [6]:
session = catalog.session(
    object_store_options=OPTIONS,
    warehouse_prefix=f"s3://{BUCKET}",
    vector_search={"nprobe": 2, "exact_overfetch": 2},
)

q = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

exact = table.search(q, top_k=3, rerank="exact", session=session)
print("exact:", exact.to_pydict())

approx = table.search(q, top_k=3, rerank="index", session=session)
print("index-only:", approx.to_pydict())

detail = table.search(
    q, top_k=3, with_distance=True, columns=["id", "title"], session=session
)
print("with distance:", detail.to_pydict())

exact: {'id': [1, 2, 6], 'title': ['doc-1', 'doc-2', 'doc-6'], 'lang': ['zh', 'zh', 'en']}
index-only: {'id': [1, 2, 6], 'title': ['doc-1', 'doc-2', 'doc-6'], 'lang': ['zh', 'zh', 'en']}
with distance: {'id': [1, 2, 6], 'title': ['doc-1', 'doc-2', 'doc-6'], '__lakesoul_vector_distance': [0.0, 0.141421374149721, 0.7071067811865476]}


[FastScan] Using AVX-512 optimized path


## 6. DataFusion SQL 查询

`LakeSoulSession.sql` 执行的就是 DataFusion 引擎；SQL 形状匹配
`ORDER BY <distance>(vec, q) LIMIT k` 时会自动下推到向量索引。

用 `session.explain` 检查计划里是否出现 `LakeSoulVectorSearchExec`。

In [ ]:
session.sql("SET lakesoul_vector_search.nprobe = 2")
session.sql("SET lakesoul_vector_search.exact_overfetch = 2")
session.sql("SET lakesoul_vector_search.rerank = 'exact'")

literal = "ARRAY[" + ", ".join(f"{value:.6g}" for value in q) + "]"
sql = f"""
SELECT id, title, array_distance(embedding, {literal}) AS distance
FROM "lakesoul"."{NAMESPACE}"."{TABLE}"
ORDER BY array_distance(embedding, {literal})
LIMIT 3
"""

print("exact:", session.sql(sql).to_pydict())

session.sql("SET lakesoul_vector_search.rerank = 'index'")
print("index-only:", session.sql(sql).to_pydict())

plan = session.explain(sql)
print("LakeSoulVectorSearchExec:", "LakeSoulVectorSearchExec" in plan)
print(plan)

exact: {'id': [1, 2, 6], 'title': ['doc-1', 'doc-2', 'doc-6'], 'distance': [0.0, 0.141421374149721, 0.7071067811865476]}
index-only: {'id': [1, 2, 6], 'title': ['doc-1', 'doc-2', 'doc-6'], 'distance': [0.0, 0.141421374149721, 0.7071067811865476]}
LakeSoulVectorSearchExec: False
+------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| plan_type                                                  | plan                                                                                                                                              

: 

## 7. 索引维护建议

- **日常写入**：`write_arrow(...)` / `INSERT` 默认自动增量维护索引，不需要手动调用；
- **历史数据首次补建**：`table.build_vector_index()`；
- **重训 / 改参数后重建**：`table.rebuild_vector_index()`，它从全部 active 文件重建并发布新 generation；
- **只重建某个分区**：`table.rebuild_vector_index(partition_desc="range=...")`；
- **只重建某一列**：`table.rebuild_vector_index(column="embedding")`。

下面代码默认不执行，按需取消注释：

```python
# table.build_vector_index(column="embedding")
# table.rebuild_vector_index()
# table.rebuild_vector_index(column="embedding", partition_desc="-5")
```

## 9. 删除 / 重建索引

本 Notebook 只演示建索引和查询。删除、重建、GC 的完整可运行流程见
[drop_vector_index.ipynb](drop_vector_index.ipynb)；参数说明见
[09-drop-index.md](09-drop-index.md)。

关键区别：

- `keep_declaration=True`：删物理索引、保留声明，之后可直接 `rebuild_vector_index()`；
- 默认 `keep_declaration=False`：声明移除，用 `recreate_vector_index()` 重新声明并重建；
- `wait_for_cleanup=False`：控制面立即删除，后台清理对象。


## 8. 清理（可选）

```python
# catalog.drop_table(TABLE, if_exists=True)
# import shutil
# shutil.rmtree(CACHE_ROOT, ignore_errors=True)
```

S3 上的数据文件和 `_vector_index/` 需要按需删除（例如用 `mc rm --recursive`）。